<a href="https://colab.research.google.com/github/Parimal1004/MyProjects/blob/main/ParimalGoudMoola_ChaitanyaBharathiInstituteOfTechnology_AI_Agent_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch transformers


In [ ]:
from transformers import pipeline
import re
import datetime
import json

# ---------------- LOADING MODEL ----------------

text_gen = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_length=128
)

print("AI Student Productivity Agent (type 'bye' to quit)")

# ---------------- MEMORY (JSON STORAGE) ----------------

TASK_FILE = "tasks.json"

def load_tasks():
    try:
        with open(TASK_FILE, "r") as f:
            data = json.load(f)


        cleaned = []
        for t in data:
            if isinstance(t, dict) and "task" in t and "priority" in t:
                if not t["task"].startswith("add task"):
                    cleaned.append(t)

        return cleaned
    except:
        return []

def save_tasks(tasks):
    with open(TASK_FILE, "w") as f:
        json.dump(tasks, f, indent=2)

tasks = load_tasks()

# ---------------- BASIC FAQ / GREETINGS ----------------

def faq_tool(user_input):
    faqs = {
        "hello": "Hello! How can I help you today?",
        "hi": "Hi there! How can I assist you?",
        "how are you": "I'm doing great! Ready to help you 😊",
        "who are you": "I am an AI Student Productivity Assistant.",
        "what can you do": (
            "I can manage tasks, create study plans, "
            "do calculations, and answer general questions."
        ),
        "thanks": "You're welcome! Happy to help.",
        "thank you": "You're welcome! 😊",
        "i am tired": "Take a short break. Even 10 minutes can help 💪",
        "i am stressed": "Try focusing on one task at a time. You’ve got this 💙",
        "motivate me": "Small steps every day lead to big success 🚀",
        "what can you do": (
        "I can manage tasks, create study plans, "
        "do calculations, tell time/date, and answer questions."
         ),
        "help": (
              "You can ask me to add tasks, view tasks, "
              "clear tasks, create a study plan, or do calculations."
          ),
        "commands": (
              "Try commands like:\n"
              "- add task study math\n"
              "- view tasks\n"
              "- study plan\n"
              "- clear tasks\n"
              "- what is the time"
       ),
    }

    text = user_input.lower()
    words = text.split()

    for key in faqs:
        if (key in text and len(key.split()) > 1) or key in words:
            return faqs[key]

    return None

# ---------------- TOOLS ----------------

def calculator_tool(text):
    try:
        parts = re.findall(r"[0-9]+|[\+\-\*/]", text)
        expression = ""
        for p in parts:
            expression += p
        value = eval(expression)
        return f"Result= {value}"
    except:
        return "I couldn't calculate that."

def time_tool():
    return "Current time: " + datetime.datetime.now().strftime("%H:%M:%S")

def date_tool():
    return "Today's date: " + datetime.datetime.now().strftime("%d-%m-%Y")

# -------- ADD TASK (RULE-BASED, CLEAN) --------

def add_task_tool(text):
    text_lower = text.lower()

    # Priority detection
    if "high priority" in text_lower or "high" in text_lower:
        priority = "high"
    elif "low priority" in text_lower or "low" in text_lower:
        priority = "low"
    else:
        priority = "medium"

    # Clean task text
    task = text_lower
    task = task.replace("add task", "")
    task = task.replace("with high priority", "")
    task = task.replace("with medium priority", "")
    task = task.replace("with low priority", "")
    task = task.strip()

    task_obj = {
        "task": task,
        "priority": priority
    }

    tasks.append(task_obj)
    save_tasks(tasks)

    return f"Task added: '{task}' with priority '{priority}'"

def view_tasks_tool():
    if not tasks:
        return "No tasks found."

    output = "Your tasks:\n"
    for t in tasks:
        output += f"- {t['task']} (Priority: {t['priority']})\n"
    return output

def clear_tasks_tool():
    tasks.clear()
    save_tasks(tasks)
    return "All tasks have been cleared successfully."

# -------- STUDY PLANNER --------

def study_planner_tool():
    if not tasks:
        return "No tasks available to create a study plan."

    plan = "Today's Study Plan:\n"

    for t in tasks:
        if t["priority"] == "high":
            plan += f"🔴 2 hours → {t['task']}\n"
        elif t["priority"] == "medium":
            plan += f"🟡 1 hour → {t['task']}\n"
        else:
            plan += f"🟢 30 minutes → {t['task']}\n"

    return plan

# -------- GENERAL CHAT --------

def chat_tool(user_input):
    prompt = f"""
You are an AI Student Productivity Assistant.

Your goals:
- Help students stay organized and productive
- Give clear, short, and practical answers
- Use simple language suitable for students
- Avoid unnecessary explanations
- Be friendly and motivating when appropriate

Rules:
- If unsure, say you are not sure instead of guessing
- Do not repeat the user's question
- Keep responses concise (2–4 lines max)

User: {user_input}
Assistant:
"""
    return text_gen(prompt)[0]["generated_text"].strip()


# ---------------- AGENT DECISION ----------------

def decide_and_act(user_input):
    # FAQ first
    faq_response = faq_tool(user_input)
    if faq_response:
        return faq_response

    decision_prompt = f"""
You are an AI agent with tools:
calculator, time, date, task manager, study planner.

Rules:
- If math → TOOL_CALCULATOR
- If asks time → TOOL_TIME
- If asks date → TOOL_DATE
- If says "add task" → TOOL_ADD_TASK
- If asks to view tasks → TOOL_VIEW_TASKS
- If says "clear tasks" → TOOL_CLEAR_TASKS
- If asks study plan → TOOL_STUDY_PLAN
- Otherwise → ANSWER

Respond with ONLY one word.

User Input: {user_input}
Decision:
"""
    decision = text_gen(decision_prompt)[0]["generated_text"].strip()

    if decision == "TOOL_CALCULATOR":
        return calculator_tool(user_input)
    elif decision == "TOOL_TIME":
        return time_tool()
    elif decision == "TOOL_DATE":
        return date_tool()
    elif decision == "TOOL_ADD_TASK":
        return add_task_tool(user_input)
    elif decision == "TOOL_VIEW_TASKS":
        return view_tasks_tool()
    elif decision == "TOOL_CLEAR_TASKS":
        return clear_tasks_tool()
    elif decision == "TOOL_STUDY_PLAN":
        return study_planner_tool()
    else:
        return chat_tool(user_input)

# ---------------- MAIN LOOP ----------------

while True:
    user_input = input("You: ")

    if user_input.lower() == "bye":
        print("Agent: Goodbye!")
        break

    response = decide_and_act(user_input)
    print("Assistant:", response)


Device set to use cpu


AI Student Productivity Agent (type 'bye' to quit)
You: hello
Assistant: Hello! How can I help you today?
You: i am tired
Assistant: Take a short break. Even 10 minutes can help 💪
You: motivate me
Assistant: Small steps every day lead to big success 🚀
You: thank you
Assistant: You're welcome! 😊
You: calculate 5*5*3
Assistant: Result= 75
You: what is the time now?
Assistant: Current time: 11:14:14
You: what is today's date?
Assistant: Today's date: 29-12-2025
You: Add task watch Stranger Things show with medium priority
Assistant: Task added: 'watch stranger things show' with priority 'medium'
You: Add task Complete IBM Project with high priority
Assistant: Task added: 'complete ibm project' with priority 'high'
You: Add task read a book with low priority
Assistant: Task added: 'read a book' with priority 'low'
You: show tasks
Assistant: Your tasks:
- watch stranger things show (Priority: medium)
- complete ibm project (Priority: high)
- read a book (Priority: low)

You: create a study 